# Apply WURST-tanh Waveforms To The NVNMR AMPM Sensing Sequence

Load the updated AMPM adiabatic-inversion-train TPS/TNT pair, write WURST-tanh phase and amplitude tables into `phmod_p180`, `phmod_pn180`, `ampmod_p180`, and `ampmod_pn180`, then write generated TPS/TNT/JSON/table artifacts. The source files are left unchanged.


## 1. Parameters

Keep operator-selected values in this cell. `f_sweep_low_hz` and `f_sweep_high_hz` define the physical WURST-tanh sweep endpoints. `freq_p180_hz` and `freq_pn180_hz` define the TNMR preset frequency shifts applied before the PhMod table. `p180_sweep_time_us` and `pn180_sweep_time_us` define the total WURST-tanh table duration, including the template's `t_mod_offset`. `p180_points` and `pn180_points` are the table lengths and are usually identical for this sequence. Leave `REASSIGN_EVENT_TABLES = False` when the template already points events at the desired tables; set it true only to repair a template.


In [ ]:
TEMPLATE_TPS_FILENAME = "20260704_NVNMR_AdiabaticInversionTrain_AMPM.tps"
TEMPLATE_TNT_FILENAME = "20260704_NVNMR_AdiabaticInversionTrain_AMPM.tnt"
OUTPUT_RELATIVE_DIR = "examples/generated/nvnmr_adiabatic_inversion_train"
OUTPUT_STEM = "20260704_NVNMR_AdiabaticInversionTrain_AMPM_wursttanh_n20_beta3_waveforms"

P180_PHASE_TABLE = "phmod_p180"
PN180_PHASE_TABLE = "phmod_pn180"
P180_AMPLITUDE_TABLE = "ampmod_p180"
PN180_AMPLITUDE_TABLE = "ampmod_pn180"
TARGET_PHASE_LINE = "F1_PhMod"
TARGET_AMPLITUDE_LINE = "F1_Ampl"
P180_EVENTS = (5, 6)
PN180_EVENTS = (12, 13)

f_sweep_low_hz = -100000.0
f_sweep_high_hz = 100000.0
freq_p180_hz = 0.0
freq_pn180_hz = 0.0
p180_sweep_time_us = 4000.0
pn180_sweep_time_us = p180_sweep_time_us
f1_amp = 100.0

p180_points = 20000
pn180_points = p180_points
WURST_ORDER = 20.0
TANH_BETA = 3.0
PHASE_STEPS = 2**16
TABLE_PRECISION = 3
REASSIGN_EVENT_TABLES = False


## 2. Setup


In [ ]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt

repo_search_start = Path.cwd().resolve()
REPO_ROOT = next(
    candidate
    for candidate in (repo_search_start, *repo_search_start.parents)
    if (candidate / "hardware_capture" / "rf" / "redstone" / "sequence.py").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REDSTONE_DIR = REPO_ROOT / "hardware_capture" / "rf" / "redstone"
OUTPUT_DIR = REDSTONE_DIR / OUTPUT_RELATIVE_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_TPS_PATH = REDSTONE_DIR / TEMPLATE_TPS_FILENAME
TEMPLATE_TNT_PATH = REDSTONE_DIR / TEMPLATE_TNT_FILENAME
OUTPUT_TPS_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tps"
OUTPUT_TNT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tnt"
OUTPUT_JSON_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tps.json"
OUTPUT_P180_PHASE_TXT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_{P180_PHASE_TABLE}_phase_table.txt"
OUTPUT_PN180_PHASE_TXT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_{PN180_PHASE_TABLE}_phase_table.txt"
OUTPUT_P180_AMPL_TXT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_{P180_AMPLITUDE_TABLE}_amplitude_table.txt"
OUTPUT_PN180_AMPL_TXT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_{PN180_AMPLITUDE_TABLE}_amplitude_table.txt"
TPS_SIGNATURE = b"PSEQ1.001.18 BIN"
TNT_EMBEDDED_SIGNATURE = b"PSEQ" + (1).to_bytes(4, "little") + b"1.18 BIN"

import hardware_capture.rf.redstone.sequence as redstone_sequence
import hardware_capture.rf.redstone.waveform as redstone_waveform

redstone_waveform = importlib.reload(redstone_waveform)
redstone_sequence = importlib.reload(redstone_sequence)
WURSTtanh = redstone_waveform.WURSTtanh
Sequence = redstone_sequence.Sequence

assert TEMPLATE_TPS_PATH.exists(), TEMPLATE_TPS_PATH
assert TEMPLATE_TNT_PATH.exists(), TEMPLATE_TNT_PATH
assert OUTPUT_TPS_PATH.resolve() != TEMPLATE_TPS_PATH.resolve()
assert OUTPUT_TNT_PATH.resolve() != TEMPLATE_TNT_PATH.resolve()


## 3. Load And Inspect The Sequence


In [ ]:
sequence = Sequence.load(TEMPLATE_TPS_PATH)
parameter_values = sequence.parameter_values()
required_parameters = {
    "f1_amp",
    "freq_p180",
    "freq_pn180",
    "t_mod_offset",
    "t_p180",
    "t_pn180",
}
missing_parameters = sorted(required_parameters - set(parameter_values))
if missing_parameters:
    raise ValueError(f"Missing required parameters: {missing_parameters}")

target_table_names = {
    P180_PHASE_TABLE,
    PN180_PHASE_TABLE,
    P180_AMPLITUDE_TABLE,
    PN180_AMPLITUDE_TABLE,
    "phmod_tail",
    "ampmod_tail",
}
target_references = [
    reference
    for reference in sequence.table_references()
    if reference.table_name in target_table_names
]
target_tables = [
    (
        table.get("name"),
        len((table.get("entries") or "").splitlines()),
        table.get("table_type_text"),
        table.get("dimension"),
    )
    for table in sequence.tables
    if table.get("name") in target_table_names
]

{
    "summary": sequence.summary().to_dict(),
    "parameters": parameter_values,
    "target_tables": target_tables,
    "target_references": target_references,
}


## 4. Helpers


In [ ]:
def duration_to_us(value: str) -> float:
    text = str(value).strip().lower()
    if text.endswith("ns"):
        return float(text[:-2]) * 1.0e-3
    if text.endswith("n"):
        return float(text[:-1]) * 1.0e-3
    if text.endswith("us"):
        return float(text[:-2])
    if text.endswith("u"):
        return float(text[:-1])
    if text.endswith("ms"):
        return float(text[:-2]) * 1.0e3
    if text.endswith("m"):
        return float(text[:-1]) * 1.0e3
    if text.endswith("s"):
        return float(text[:-1]) * 1.0e6
    return float(text)


def embed_tps_in_tnt(
    *,
    template_tps_path: Path,
    edited_tps_path: Path,
    template_tnt_path: Path,
    output_tnt_path: Path,
) -> int:
    template_tps_bytes = template_tps_path.read_bytes()
    edited_tps_bytes = edited_tps_path.read_bytes()
    template_tnt_bytes = template_tnt_path.read_bytes()

    if not template_tps_bytes.startswith(TPS_SIGNATURE):
        raise ValueError("Template TPS has an unexpected PSEQ signature.")
    if not edited_tps_bytes.startswith(TPS_SIGNATURE):
        raise ValueError("Edited TPS has an unexpected PSEQ signature.")

    template_embedded = TNT_EMBEDDED_SIGNATURE + template_tps_bytes[16:]
    edited_embedded = TNT_EMBEDDED_SIGNATURE + edited_tps_bytes[16:]
    start = template_tnt_bytes.find(template_embedded)
    if start < 0:
        starts = [
            index
            for index in range(len(template_tnt_bytes))
            if template_tnt_bytes.startswith(TNT_EMBEDDED_SIGNATURE, index)
        ]
        if len(starts) != 1:
            raise ValueError("The template TNT does not contain a unique embedded TPS payload.")
        start = starts[0]
        embedded_text = TPS_SIGNATURE + template_tnt_bytes[
            start + len(TNT_EMBEDDED_SIGNATURE) : start + len(template_embedded)
        ]
        Sequence.from_tps_bytes(embedded_text)

    output_tnt_path.write_bytes(
        template_tnt_bytes[:start]
        + edited_embedded
        + template_tnt_bytes[start + len(template_embedded) :]
    )

    restored_embedded = output_tnt_path.read_bytes()[start : start + len(edited_embedded)]
    if TPS_SIGNATURE + restored_embedded[16:] != edited_tps_bytes:
        raise ValueError("Embedded TNT TPS payload did not round-trip.")
    return start


## 5. Build The WURST-tanh Waveforms


In [ ]:
p180_points = int(p180_points)
pn180_points = int(pn180_points)
if p180_points <= 0 or pn180_points <= 0:
    raise ValueError("p180_points and pn180_points must be positive integers.")
t_mod_offset_us = duration_to_us(parameter_values["t_mod_offset"])
p180_duration_us = float(p180_sweep_time_us)
pn180_duration_us = float(pn180_sweep_time_us)
if p180_duration_us <= t_mod_offset_us or pn180_duration_us <= t_mod_offset_us:
    raise ValueError("Sweep times must be greater than t_mod_offset.")

f_sweep_low_hz = float(f_sweep_low_hz)
f_sweep_high_hz = float(f_sweep_high_hz)
freq_p180_hz = float(freq_p180_hz)
freq_pn180_hz = float(freq_pn180_hz)
sweep_width_hz = f_sweep_high_hz - f_sweep_low_hz
if sweep_width_hz <= 0:
    raise ValueError("Expected f_sweep_high_hz to be greater than f_sweep_low_hz.")
f_sweep_center_hz = 0.5 * (f_sweep_low_hz + f_sweep_high_hz)

f1_amp = float(f1_amp)
p180_waveform = WURSTtanh(
    points=p180_points,
    sweep_width_hz=sweep_width_hz,
    dwell_s=p180_duration_us * 1.0e-6 / p180_points,
    order=WURST_ORDER,
    beta=TANH_BETA,
    amp=f1_amp,
    center_hz=f_sweep_center_hz,
    offset_hz=freq_p180_hz,
    phase_steps=PHASE_STEPS,
)
pn180_waveform = WURSTtanh(
    points=pn180_points,
    sweep_width_hz=-sweep_width_hz,
    dwell_s=pn180_duration_us * 1.0e-6 / pn180_points,
    order=WURST_ORDER,
    beta=TANH_BETA,
    amp=f1_amp,
    center_hz=f_sweep_center_hz,
    offset_hz=freq_pn180_hz,
    phase_steps=PHASE_STEPS,
)

p180_frequency = p180_waveform.frequency_values()
pn180_frequency = pn180_waveform.frequency_values()
p180_phmod_frequency = [value - p180_waveform.offset_hz for value in p180_frequency]
pn180_phmod_frequency = [value - pn180_waveform.offset_hz for value in pn180_frequency]

{
    "p180": p180_waveform.summary().to_dict(),
    "pn180": pn180_waveform.summary().to_dict(),
    "p180_total_frequency_span_hz": (
        p180_frequency[0],
        p180_frequency[-1],
    ),
    "pn180_total_frequency_span_hz": (
        pn180_frequency[0],
        pn180_frequency[-1],
    ),
    "p180_phmod_frequency_span_hz": (
        p180_phmod_frequency[0],
        p180_phmod_frequency[-1],
    ),
    "pn180_phmod_frequency_span_hz": (
        pn180_phmod_frequency[0],
        pn180_phmod_frequency[-1],
    ),
    "f_sweep_center_hz": f_sweep_center_hz,
    "p180_sequence_offset_hz": p180_waveform.offset_hz,
    "pn180_sequence_offset_hz": pn180_waveform.offset_hz,
    "t_mod_offset_us": t_mod_offset_us,
    "p180_duration_us": p180_duration_us,
    "pn180_duration_us": pn180_duration_us,
}


## 6. Visualize The WURST-tanh Tables


In [ ]:
p180_time_us = [index * p180_waveform.dwell_s * 1.0e6 for index in range(p180_waveform.points)]
pn180_time_us = [index * pn180_waveform.dwell_s * 1.0e6 for index in range(pn180_waveform.points)]
p180_amp = p180_waveform.amplitude_values()
pn180_amp = pn180_waveform.amplitude_values()
p180_phase = p180_waveform.phase_values()
pn180_phase = pn180_waveform.phase_values()

fig, axes = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
axes[0].plot(p180_time_us, p180_amp, label=f"p180 {P180_AMPLITUDE_TABLE}")
axes[0].plot(pn180_time_us, pn180_amp, "--", label=f"pn180 {PN180_AMPLITUDE_TABLE}")
axes[0].set_ylabel("Amplitude")
axes[0].legend()

axes[1].plot(p180_time_us, p180_frequency, label="p180 absolute")
axes[1].plot(pn180_time_us, pn180_frequency, label="pn180 absolute")
axes[1].set_ylabel("RF offset (Hz)")
axes[1].legend()

axes[2].plot(p180_time_us, p180_phmod_frequency, label=f"p180 {P180_PHASE_TABLE} dphase/dt")
axes[2].plot(pn180_time_us, pn180_phmod_frequency, label=f"pn180 {PN180_PHASE_TABLE} dphase/dt")
axes[2].set_ylabel("PhMod offset (Hz)")
axes[2].legend()

axes[3].plot(p180_time_us, p180_phase, label=P180_PHASE_TABLE)
axes[3].plot(pn180_time_us, pn180_phase, label=PN180_PHASE_TABLE)
axes[3].set_xlabel("Time (us)")
axes[3].set_ylabel("Wrapped TNMR phase (deg)")
axes[3].legend()

fig.tight_layout()


## 7. Apply WURST-tanh Tables And Write The TPS


In [ ]:
sequence.set_waveform_to_table(
    P180_PHASE_TABLE,
    p180_waveform,
    quantity="phase",
    precision=TABLE_PRECISION,
    phase_units="deg",
    allow_shared=True,
)
sequence.set_waveform_to_table(
    PN180_PHASE_TABLE,
    pn180_waveform,
    quantity="phase",
    precision=TABLE_PRECISION,
    phase_units="deg",
    allow_shared=True,
)
sequence.set_waveform_to_table(
    P180_AMPLITUDE_TABLE,
    p180_waveform,
    quantity="amplitude",
    precision=TABLE_PRECISION,
    allow_shared=True,
)
sequence.set_waveform_to_table(
    PN180_AMPLITUDE_TABLE,
    pn180_waveform,
    quantity="amplitude",
    precision=TABLE_PRECISION,
    allow_shared=True,
)

expected_event_tables = set()
for event_number in P180_EVENTS:
    expected_event_tables.add((TARGET_PHASE_LINE, event_number, P180_PHASE_TABLE))
    expected_event_tables.add((TARGET_AMPLITUDE_LINE, event_number, P180_AMPLITUDE_TABLE))
for event_number in PN180_EVENTS:
    expected_event_tables.add((TARGET_PHASE_LINE, event_number, PN180_PHASE_TABLE))
    expected_event_tables.add((TARGET_AMPLITUDE_LINE, event_number, PN180_AMPLITUDE_TABLE))
if REASSIGN_EVENT_TABLES:
    for event_number in P180_EVENTS:
        sequence.assign_table_to_event(TARGET_PHASE_LINE, event_number, P180_PHASE_TABLE)
        sequence.assign_table_to_event(TARGET_AMPLITUDE_LINE, event_number, P180_AMPLITUDE_TABLE)
    for event_number in PN180_EVENTS:
        sequence.assign_table_to_event(TARGET_PHASE_LINE, event_number, PN180_PHASE_TABLE)
        sequence.assign_table_to_event(TARGET_AMPLITUDE_LINE, event_number, PN180_AMPLITUDE_TABLE)

sequence.stored_sequence_path = OUTPUT_STEM
sequence.set_parameter_value("freq_p180", f"{freq_p180_hz:.15g}")
sequence.set_parameter_value("freq_pn180", f"{freq_pn180_hz:.15g}")
sequence.set_parameter_value("f1_amp", f"{f1_amp:.15g}")
sequence.set_parameter_value("t_p180", f"{p180_duration_us - t_mod_offset_us:.15g}u")
sequence.set_parameter_value("t_pn180", f"{pn180_duration_us - t_mod_offset_us:.15g}u")

p180_waveform.write_phase_column(OUTPUT_P180_PHASE_TXT_PATH, precision=TABLE_PRECISION)
pn180_waveform.write_phase_column(OUTPUT_PN180_PHASE_TXT_PATH, precision=TABLE_PRECISION)
p180_waveform.write_amplitude_column(OUTPUT_P180_AMPL_TXT_PATH, precision=TABLE_PRECISION)
pn180_waveform.write_amplitude_column(OUTPUT_PN180_AMPL_TXT_PATH, precision=TABLE_PRECISION)
sequence.write_json(OUTPUT_JSON_PATH, include_raw_bytes=False)
sequence.write_tps(OUTPUT_TPS_PATH)

reloaded = Sequence.load(OUTPUT_TPS_PATH)
reloaded_tables = {table.get("name"): table for table in reloaded.tables}
reloaded_event_tables = {
    (reference.line, reference.event_number, reference.table_name)
    for reference in reloaded.table_references()
}
missing_event_tables = sorted(expected_event_tables - reloaded_event_tables)
if missing_event_tables:
    raise ValueError(
        "The sequence is missing expected WURST-tanh table bindings: "
        f"{missing_event_tables}. Set REASSIGN_EVENT_TABLES=True only when repairing a template."
    )
expected_tables = {
    P180_PHASE_TABLE: p180_points,
    PN180_PHASE_TABLE: pn180_points,
    P180_AMPLITUDE_TABLE: p180_points,
    PN180_AMPLITUDE_TABLE: pn180_points,
}
for table_name, point_count in expected_tables.items():
    actual_count = len((reloaded_tables[table_name].get("entries") or "").splitlines())
    if actual_count != point_count:
        raise ValueError(f"Table {table_name!r} has {actual_count} rows, expected {point_count}.")

{
    "output_tps": OUTPUT_TPS_PATH,
    "output_json": OUTPUT_JSON_PATH,
    "tables": {
        name: len((reloaded_tables[name].get("entries") or "").splitlines())
        for name in expected_tables
    },
    "event_table_bindings_reassigned": REASSIGN_EVENT_TABLES,
}


## 8. Embed The Edited TPS Into The TNT


In [ ]:
embedded_offset = embed_tps_in_tnt(
    template_tps_path=TEMPLATE_TPS_PATH,
    edited_tps_path=OUTPUT_TPS_PATH,
    template_tnt_path=TEMPLATE_TNT_PATH,
    output_tnt_path=OUTPUT_TNT_PATH,
)

edited_tps_bytes = OUTPUT_TPS_PATH.read_bytes()
edited_embedded = TNT_EMBEDDED_SIGNATURE + edited_tps_bytes[16:]
output_tnt_bytes = OUTPUT_TNT_PATH.read_bytes()
assert output_tnt_bytes[embedded_offset : embedded_offset + len(edited_embedded)] == edited_embedded

{
    "output_tnt": OUTPUT_TNT_PATH,
    "tnt_bytes": OUTPUT_TNT_PATH.stat().st_size,
    "tps_bytes": OUTPUT_TPS_PATH.stat().st_size,
    "embedded_tps_offset": embedded_offset,
}


## 9. Generated Files


In [ ]:
generated_files = [
    OUTPUT_TPS_PATH,
    OUTPUT_TNT_PATH,
    OUTPUT_JSON_PATH,
    OUTPUT_P180_PHASE_TXT_PATH,
    OUTPUT_PN180_PHASE_TXT_PATH,
    OUTPUT_P180_AMPL_TXT_PATH,
    OUTPUT_PN180_AMPL_TXT_PATH,
]
[{"file": path.name, "path": str(path), "bytes": path.stat().st_size} for path in generated_files]
